<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 10 · Linear and Generalized Linear Models for Return Prediction

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
Chapter 10 focuses on linear/GLM baselines. Here we:
- engineer cross-sectional features,
- fit ridge/lasso/logistic models with scikit-learn,
- evaluate IC/AUC metrics, and
- translate predictions into portfolio weights.


### Getting Help While Modeling
- **Appendix B** for pandas feature engineering patterns.
- **Appendix C** for scikit-learn estimator syntax.
- **Chapter 8** for performance metric definitions.

ROC-AUC score: 真正会上涨的股票，是否整体上比不会上涨的股票得分更高。

ROC是什么？  
当不断移动阈值时，真阳性TPR和假阳性FPR如何变化？  
TPR: 真正上涨时，模型抓住多少？   
    TPR = TP / (TP + FN)   
FPR: 实际没涨时，模型误判为阳的比例是多少？  
    FPR = FP / (FP + TN)  

应用：每次移动阈值（阈值是模型从概率预测到分类的边界线），都对应着一对(TPR, FPR)  

AUC = 0.6已经很厉害了。  

工作中，
step1. AUC/IC 是否稳定；  
step2. 不同regime是否稳定；  
step3. 做：  

1. decile analysis 分层分析，排名前面的股票， 是否真的比后面的更赚钱。

2. turnover analysis 换手率分析，手续费、滑点、冲击成本、bid-ask spread，所以要关注单位 alpha 对应多少 turnover。

3.  factor exposure：
做因子回归： strategy_return
alpha = beta1 * market + beta2 * size + beta3 * value  

step4. 进入组合层：
考虑 1.仓位； 2.alpha相关性；3.风险预算（职业世界里分配的不是资金，而是风险）；4. 市场状态 Regime switching.

【感悟】：市场不是一个预测问题，而是动态生存系统，再强的alhpa也会有decay，再好的model也会有regime break，真正活下来的是适应能力最强的人，要构建一个系统/生态才对。


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, roc_auc_score

## 1. Build Cross-Sectional Feature Panel

This section reshapes the time‑series price panel into a stacked (date, asset) structure with momentum and volatility features for every asset‑date pair.

### 1.0 Data Loading
We load prices once for the cross-sectional panel.

In [16]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

In [ ]:
panel = prices.ffill().pct_change()
features = panel.rolling(20).mean()
vol = panel.rolling(20).std()
X = pd.concat({"mom": features, "vol": vol}, axis=1).stack().dropna()  # concat只包含给的列的拼接，不包含原来dataframe中的列。
y = panel.shift(-1).stack().reindex(X.index).dropna()
X = X.loc[y.index]
data = pd.DataFrame({"mom": X["mom"], "vol": X["vol"], "label": y})
data.head()

In [32]:
mom = features.stack()
volatility = vol.stack()

label = panel.shift(-1).stack()

data2 = pd.DataFrame({
    "mom": mom,
    "vol": volatility,
    "label": label
}).dropna()

data2


mom       vol       label
Date                                              
2015-12-29 AAPL    -0.004089  0.015565   24.197300
           BTC-USD  0.007689  0.040920  426.619995
           EURUSD   0.001418  0.008586    1.086900
           GLD      0.000196  0.011148  101.420000
           JPM      0.000434  0.017354   50.808600
...                      ...       ...         ...
2025-11-26 GLD      0.002767  0.011839  387.880000
           JPM      0.000429  0.013045  313.080000
           NVDA    -0.006569  0.026537  177.000000
           SPY     -0.000519  0.009678  683.390000
           TLT     -0.000055  0.005004   90.210000

[19952 rows x 3 columns]

### 1.1 Train/Test Split by Date

We cut the panel into an early training segment and a later test segment along the calendar axis to mimic a realistic research workflow.

In [ ]:
dates = data.index.get_level_values(0)
split_idx = int(len(dates) * 0.7)
split_date = dates.sort_values()[split_idx]
train = data.loc[dates <= split_date]         # 取出日期 <= split_date 的所有行
test = data.loc[dates > split_date]
X_train, y_train = train[["mom", "vol"]], train["label"]   # 纵向切
X_test, y_test = test[["mom", "vol"]], test["label"]

## 2. Ridge vs. Lasso Regression

We compare ridge and lasso models on the same feature set to see how L2 vs. L1 regularization affects fit quality.

In [40]:
ridge = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=5.0)),
])
lasso = Pipeline([
    ("scale", StandardScaler()),
    ("model", Lasso(alpha=0.001)),
])
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
lasso_pred = lasso.predict(X_test)
pd.Series({"ridge_mse": mean_squared_error(y_test, ridge_pred), "lasso_mse":
mean_squared_error(y_test, lasso_pred)})

,0
ridge_mse,0.000344
lasso_mse,0.000345


## 3. Logistic Regression for Directional Bets

Here we reframe the prediction target as up/down moves and fit a logistic regression that outputs probabilities instead of raw returns.

In [41]:
y_train_cls = (y_train > 0).astype(int)     # 需要将label变换一下。
y_test_cls = (y_test > 0).astype(int)
logit = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
logit.fit(X_train, y_train_cls)
logit_prob = logit.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test_cls, logit_prob)   # 返回AUC的值，表示模型给给证样本打得分或者概率 > 负样本的概率。
auc

np.float64(0.5161087398158415)

### Performance Helper
We reuse the Chapter 8 metric utility here for completeness.

In [44]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()        #
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_ret,
            "annualized_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
        }
    )

### 3.1 Portfolio Signal from Probabilities

We turn predicted probabilities into position sizes and evaluate the resulting long/short strategy using the performance helper.

In [45]:
weights = pd.Series(logit_prob - 0.5, index=y_test_cls.index).clip(-0.5, 0.5)  # 做权重的缩尾
asset_returns = pd.Series(y_test.values, index=y_test.index)
strategy = weights * asset_returns
performance_stats(strategy)
wealth

NameError: name 'wealth' is not defined

## 4. Exercises
### Exercise 1 – Elastic Net
Add an Elastic Net model to the comparison table.
<details><summary>Hint</summary>
Use <code>sklearn.linear_model.ElasticNet</code> with <code>l1_ratio</code> grid search.
</details>

### Exercise 2 – Feature Scaling Variants
Experiment with <code>RobustScaler</code> and compare coefficients.
<details><summary>Hint</summary>
Swap the scaler step inside the pipeline and refit models.
</details>

### Exercise 3 – Threshold Tuning
Test different probability thresholds (0.4/0.6, etc.) when generating portfolio weights.
<details><summary>Hint</summary>
Change the subtraction from 0.5 to the chosen cutoffs and recompute <code>performance_stats</code>.
</details>


## 5. Takeaways for Chapter 10
- Feature stacking turns time-series data into cross-sectional ML inputs.
- Pipelines with scalers keep training reproducible.
- Logistic probabilities make it easy to translate classification outputs into weights.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">